This notebook prcesses raw skeletons by breaking branches, and remerging the fragments. After all strips from a given directory are processed, they are then combined using translation and offset information.

In [ ]:
import zarr
import numpy as np
import matplotlib.pyplot as plt
import navis
import os
import requests
import glob
from pathlib import Path
from io import BytesIO

import concurrent.futures
from joblib import dump,load, Parallel, delayed, parallel_config
from natsort import natsorted
import uuid
import random
import pandas as pd
from acanalysis.skeleton_reconstruction.reconnect_skeletons import *
from scipy.spatial import distance
from acanalysis.skeleton_reconstruction.util import read_navis_neurons_tar, write_navis_skels_tar, remove_cutout_nodes, translate_nodes, create_rectangle_volume, remove_overlap_nodes, filter_skeletons
from acanalysis.skeleton_reconstruction.interstrip_reconnect import *

In [ ]:
#set parameters
n_jobs = 10 #number of parallel jobs
im_shape = [576,576,45696] #strip shape
sc = "/ACdata/Users/connorl/Models/scaler.joblib" #scalar file
cl = "/ACdata/Users/connorl/Models/LR_1.joblib" #model file
out_dir = "./"

#Load translation data
file_trans = pd.read_json("S32_file_trans.json", orient='records', lines=True)
file_trans.columns = ['File','Translation']

#Match original files to output files
out_fn = {}
for ind,row in file_trans.iterrows():
    rem = os.path.dirname(row['File'])+"/"
    f = row['File'].replace(rem, out_dir)
    out_fn[row['File']] = f

In [ ]:
#find overlapping volumes
files, translations = list(file_trans['File']), list(file_trans['Translation'])
matches,volumes = find_overlap_volumes(files=files, translations=translations , im_shape=im_shape)

In [ ]:
#order the matches so there are no duplicate files in a job batch
matches =  order_matches(matches, 10)

In [ ]:
#find boundaries of overlap
bounds = find_overlap_bounds(matches,volumes)

In [ ]:
#translate and filter individual strips (assuming individual strips have already had reconnection run)
with parallel_config(backend="loky", inner_max_num_threads=1):
    %time res = Parallel(n_jobs=15)(delayed(postprocess_strip)(out_dir=out_dir, file=row['File'], cl=cl, sc=sc, bound_boxs=bounds[row['File']], trans=row['Translation']) for ind,row in file_trans.iterrows())

In [ ]:
#reconnect adjacent strips
with parallel_config(backend="loky", inner_max_num_threads=1):
    %time res = Parallel(n_jobs=5)(delayed(reconnect_strips)(strip1=out_fn[row['File1']] , strip2=out_fn[row['File2']], overlap=row['xyz_overlap'], cl=cl, sc=sc) for ind,row in matches.iterrows())

In [ ]:
#pull out merged skeletons
with parallel_config(backend="loky", inner_max_num_threads=1):
    %time merged_skels = Parallel(n_jobs=6)(delayed(get_merged_skeletons)(file=out_fn[row['File']]) for ind,row in file_trans.iterrows())

In [ ]:
#remove overlapping nodes and run final reconnection
merged_skels = navis.NeuronList(merged_skels)

In [ ]:
%time overlap_rem = remove_overlap_nodes(merged_skels.copy())

In [ ]:
#run final reconnection
%time non_merged,merged =  reconnect(skels=overlap_rem, cl=cl, sc=sc, min_nodes=0, query_dis=10, min_collin=0.7, prob_thresh=.1, downsample=None, smooth=None, split=False)
write_navis_skels_tar(out_dir+'merged.swcs.tar.gz', navis.NeuronList([non_merged,merged]))